# 10 — ML Meta-Label Dataset

Bu aşamada Robot stratejisini değiştirmiyoruz. Makine öğrenmesinin görevi:

> Robotun ürettiği bir AL sinyalinin alınmaya değer olup olmadığını tahmin etmek.

Her satır bir Robot giriş olayıdır. Etiket, final stratejinin aynı çıkış
kuralları ve maliyetleriyle elde edilen net işlem sonucundan oluşturulur.

Önemli kurallar:

- Özellikler yalnızca sinyal günü ve geçmiş verileri kullanır.
- Giriş bir sonraki işlem günü açılışındadır.
- Aynı hissede bir işlem açıkken yeni örnek üretilmez.
- Portföy limiti uygulanmaz; amaç bütün birincil sinyal olaylarını öğrenmektir.
- `2025+` dönemi daha önce görüldüğü için gerçek anlamda dokunulmamış test
  değildir. Gerçek yeni out-of-sample doğrulama paper trading olacaktır.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src").is_dir()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%load_ext autoreload
%autoreload 2

from src.features import add_indicators
from src.signals import (
    build_market_regime,
    add_robot_scores,
)
from src.presets import (
    FINAL_STRATEGY_CONFIG,
    FINAL_PORTFOLIO_CONFIG,
)
from src.ml_dataset import (
    BASE_FEATURE_COLUMNS,
    TARGET_COLUMNS,
    add_meta_features,
    build_meta_label_dataset,
    leakage_audit,
    save_feature_manifest,
)


## 1. Temiz verileri yükle


In [ ]:
stock_prices = pd.read_parquet(
    PROJECT_ROOT
    / "data"
    / "processed"
    / "bist100_robot_clean.parquet"
)

market_prices = pd.read_parquet(
    PROJECT_ROOT
    / "data"
    / "processed"
    / "xu100_robot_clean.parquet"
)

print("Hisse verisi:", stock_prices.shape)
print("Endeks verisi:", market_prices.shape)
print(
    "Tarih aralığı:",
    stock_prices["Date"].min(),
    "→",
    stock_prices["Date"].max(),
)


## 2. Final Robot skorlarını ve ML özelliklerini üret


In [ ]:
stock_features = add_indicators(stock_prices)
market_features = add_indicators(market_prices)
market_regime = build_market_regime(market_features)

scored_prices = add_robot_scores(
    stock_features=stock_features,
    market_regime=market_regime,
    config=FINAL_STRATEGY_CONFIG,
    include_reasons=True,
)

featured_prices = add_meta_features(
    scored_prices=scored_prices,
    market_features=market_features,
)

print("Özellikli satır:", len(featured_prices))
print("Özellik sayısı:", len(BASE_FEATURE_COLUMNS))
print("AL sinyali satırı:", featured_prices["Signal"].eq("AL").sum())


## 3. Stratejiyle uyumlu olay etiketlerini oluştur


In [ ]:
ml_dataset = build_meta_label_dataset(
    featured_prices=featured_prices,
    market_features=market_features,
    strategy_config=FINAL_STRATEGY_CONFIG,
    portfolio_config=FINAL_PORTFOLIO_CONFIG,
    feature_columns=BASE_FEATURE_COLUMNS,
)

print("Toplam olay:", len(ml_dataset))
print("Hisse sayısı:", ml_dataset["Ticker"].nunique())
print(
    "Sinyal tarih aralığı:",
    ml_dataset["Signal_Date"].min(),
    "→",
    ml_dataset["Signal_Date"].max(),
)

display(ml_dataset.head())


## 4. Sansürlü ve aykırı örnekleri ayır

Verinin sonuna kadar kapanmamış işlemler `Is_Censored=True` olur. Bunlar
gelecekteki sonucu bilinmediği için model eğitiminde kullanılmamalıdır.


In [ ]:
training_dataset = (
    ml_dataset.loc[
        ~ml_dataset["Is_Censored"]
        & ~ml_dataset["Is_Outlier"]
    ]
    .copy()
    .reset_index(drop=True)
)

excluded_summary = pd.DataFrame(
    [
        {
            "Total_Events": len(ml_dataset),
            "Censored_Events": int(
                ml_dataset["Is_Censored"].sum()
            ),
            "Outlier_Events": int(
                ml_dataset["Is_Outlier"].sum()
            ),
            "Training_Eligible_Events": len(
                training_dataset
            ),
        }
    ]
)

display(excluded_summary)


## 5. Veri sızıntısı ve yapısal kontroller


In [ ]:
leakage_report = leakage_audit(
    training_dataset
)

display(leakage_report)

if not leakage_report["Passed"].all():
    raise RuntimeError(
        "Leakage audit başarısız. Model aşamasına geçme."
    )


## 6. Dönem ve sınıf dağılımı


In [ ]:
split_summary = (
    training_dataset.groupby("Period")
    .agg(
        Event_Count=("Meta_Label", "size"),
        Positive_Count=("Meta_Label", "sum"),
        Positive_Rate=("Meta_Label", "mean"),
        Average_Return_=("Net_Return_%", "mean"),
        Median_Return_=("Net_Return_%", "median"),
        Average_R_Multiple=("R_Multiple", "mean"),
        Average_Holding_Bars=("Holding_Bars", "mean"),
    )
    .reset_index()
)

split_summary["Positive_Rate_%"] = (
    split_summary["Positive_Rate"] * 100
)

display(split_summary)


In [ ]:
class_balance = (
    training_dataset.groupby(
        ["Period", "Meta_Label"]
    )
    .size()
    .rename("Count")
    .reset_index()
)

class_balance["Class"] = class_balance[
    "Meta_Label"
].map(
    {
        0: "Başarısız sinyal",
        1: "Başarılı sinyal",
    }
)

display(class_balance)


## 7. Sonuçların skor ve çıkış nedenine göre dağılımı


In [ ]:
outcome_by_score = (
    training_dataset.groupby("Score")
    .agg(
        Event_Count=("Meta_Label", "size"),
        Win_Rate=("Meta_Label", "mean"),
        Average_Return_=("Net_Return_%", "mean"),
        Median_Return_=("Net_Return_%", "median"),
        Average_R_Multiple=("R_Multiple", "mean"),
    )
    .reset_index()
)

outcome_by_score["Win_Rate_%"] = (
    outcome_by_score["Win_Rate"] * 100
)

display(outcome_by_score)


In [ ]:
exit_reason_summary = (
    training_dataset.groupby("Exit_Reason")
    .agg(
        Event_Count=("Meta_Label", "size"),
        Win_Rate=("Meta_Label", "mean"),
        Average_Return_=("Net_Return_%", "mean"),
        Median_Return_=("Net_Return_%", "median"),
        Average_Holding_Bars=("Holding_Bars", "mean"),
    )
    .sort_values("Event_Count", ascending=False)
    .reset_index()
)

exit_reason_summary["Win_Rate_%"] = (
    exit_reason_summary["Win_Rate"] * 100
)

display(exit_reason_summary)


## 8. Yıllara göre örnek sayısı ve başarı oranı


In [ ]:
yearly_summary = (
    training_dataset.assign(
        Year=training_dataset["Signal_Date"].dt.year
    )
    .groupby("Year")
    .agg(
        Event_Count=("Meta_Label", "size"),
        Win_Rate=("Meta_Label", "mean"),
        Average_Return_=("Net_Return_%", "mean"),
        Median_Return_=("Net_Return_%", "median"),
    )
    .reset_index()
)

yearly_summary["Win_Rate_%"] = (
    yearly_summary["Win_Rate"] * 100
)

display(yearly_summary)


In [ ]:
plt.figure(figsize=(12, 6))
plt.bar(
    yearly_summary["Year"].astype(str),
    yearly_summary["Event_Count"],
)
plt.title("Yıllara Göre Meta-Label Olay Sayısı")
plt.xlabel("Yıl")
plt.ylabel("Olay Sayısı")
plt.tight_layout()
plt.show()


## 9. Özellik eksikliği ve temel istatistikler


In [ ]:
feature_missingness = (
    training_dataset[BASE_FEATURE_COLUMNS]
    .isna()
    .sum()
    .rename("Missing_Count")
    .reset_index()
    .rename(columns={"index": "Feature"})
)

feature_missingness["Missing_Rate_%"] = (
    feature_missingness["Missing_Count"]
    / len(training_dataset)
    * 100
)

display(
    feature_missingness.sort_values(
        "Missing_Count",
        ascending=False,
    )
)


In [ ]:
feature_describe = (
    training_dataset[BASE_FEATURE_COLUMNS]
    .describe()
    .T
)

display(feature_describe)


## 10. Dosyaları kaydet

Model notebook'u yalnızca `training_dataset` dosyasını kullanacaktır.
Tam olay tablosu, sansürlü son işlemlerin daha sonra güncellenebilmesi için
ayrıca saklanır.


In [ ]:
ML_RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
    / "ml"
)
ML_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

ml_dataset.to_parquet(
    ML_RESULTS_DIR
    / "robot_meta_label_events_all.parquet",
    index=False,
)

training_dataset.to_parquet(
    ML_RESULTS_DIR
    / "robot_meta_label_training.parquet",
    index=False,
)

training_dataset.to_csv(
    ML_RESULTS_DIR
    / "robot_meta_label_training.csv",
    index=False,
)

split_summary.to_csv(
    ML_RESULTS_DIR
    / "meta_label_split_summary.csv",
    index=False,
)

outcome_by_score.to_csv(
    ML_RESULTS_DIR
    / "meta_label_outcome_by_score.csv",
    index=False,
)

exit_reason_summary.to_csv(
    ML_RESULTS_DIR
    / "meta_label_exit_reasons.csv",
    index=False,
)

yearly_summary.to_csv(
    ML_RESULTS_DIR
    / "meta_label_yearly_summary.csv",
    index=False,
)

leakage_report.to_csv(
    ML_RESULTS_DIR
    / "meta_label_leakage_audit.csv",
    index=False,
)

save_feature_manifest(
    feature_columns=BASE_FEATURE_COLUMNS,
    path=MODEL_DIR
    / "meta_label_feature_manifest.json",
)

print("ML meta-label dataset dosyaları kaydedildi.")


## Sonraki aşama

Model seçiminde rastgele train-test split kullanılmayacak.

Plan:

1. Development döneminde zaman sıralı eğitim ve cross-validation
2. Validation döneminde model ve olasılık eşiği seçimi
3. `2025+` döneminde yalnızca raporlama
4. Seçilen ML filtresini final portföy backtestine entegre etme
5. Robot + ML sonucunu hem orijinal Robot hem de BIST100 ile karşılaştırma
